<a href="https://colab.research.google.com/github/matiascardacce/DL_Matias_Cardacce/blob/main/Copia_de_TP_clase3_clasificacion_corrosion_yolov8_Matias_Cardacce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Clase 3 — Clasificación de imágenes con YOLOv8
## Caso: detección de corrosión en superficies metálicas

**Alumno:** Matías Cardacce — Diplomatura en IA, UTN

Adaptación del notebook de la clase (clasificación YOLOv8 sobre dataset personalizado) a un caso de inspección de activos metálicos.

**Dataset:** `khoaliamle/Corrosion_Rust` (Hugging Face Hub, licencia MIT) — 752 imágenes en dos clases: *rust* / *no rust*. Se descarga sin credenciales y se reorganiza en `train/val/test` con una carpeta por clase, que es el formato que espera `yolo task=classify`.

**Antes de arrancar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

**Flujo:** verificar GPU → instalar Ultralytics → descargar y particionar el dataset → entrenar → validar → evaluar sobre test → conclusiones.

> El notebook está escrito para que, cambiando únicamente la celda de descarga por un `unzip` de fotos propias con la misma estructura de carpetas, todo lo demás funcione igual.

## 1. Verificación del entorno

In [ ]:
!nvidia-smi

In [ ]:
import os

HOME = os.getcwd()
print('HOME:', HOME)

In [ ]:
# Instalación de Ultralytics (sin pin de versión, para evitar conflictos con el torch/numpy actual de Colab).
# Si Colab pide 'Restart session', reiniciá y volvé a correr DESDE LA CELDA DE HOME (el pip no hace falta repetirlo).

!pip install ultralytics huggingface_hub -q
!yolo settings sync=False

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

## 2. Descarga del dataset

El repositorio viene con las imágenes agrupadas por clase pero **sin partición**:

```
Corrosion/
├── rust/       (346 imágenes)
└── no rust/    (406 imágenes)
```

Son ~100 MB, la descarga tarda menos de un minuto.

In [ ]:
from huggingface_hub import snapshot_download

ORIGEN = snapshot_download(
    repo_id='khoaliamle/Corrosion_Rust',
    repo_type='dataset',
    local_dir='/content/raw_corrosion'
)

ORIGEN = os.path.join(ORIGEN, 'Corrosion')
print('Descargado en:', ORIGEN)
print('Clases originales:', [c for c in sorted(os.listdir(ORIGEN)) if os.path.isdir(os.path.join(ORIGEN, c))])

## 3. Partición en train / val / test

Se arma la estructura que necesita YOLO, con nombres de clase en castellano:

```
datasets/corrosion/
├── train/   70 %
│   ├── con_corrosion/
│   └── sin_corrosion/
├── val/     20 %
└── test/    10 %
```

El split es aleatorio con semilla fija (`seed=42`) para que el resultado sea reproducible. Antes de copiar, cada archivo se abre con PIL: las imágenes corruptas se descartan en lugar de hacer fallar el entrenamiento a mitad de camino.

In [ ]:
import glob, random, shutil
from PIL import Image as PILImage

random.seed(42)

DATASET = '/content/datasets/corrosion'
MAPA = {'rust': 'con_corrosion', 'no rust': 'sin_corrosion'}
EXTENSIONES = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

shutil.rmtree(DATASET, ignore_errors=True)

descartadas = 0

for carpeta_origen, clase in MAPA.items():
    archivos = [f for f in glob.glob(os.path.join(ORIGEN, carpeta_origen, '*'))
                if f.lower().endswith(EXTENSIONES)]

    validos = []
    for f in archivos:
        try:
            with PILImage.open(f) as im:
                im.verify()
            validos.append(f)
        except Exception:
            descartadas += 1

    random.shuffle(validos)
    n = len(validos)
    n_train = int(n * 0.70)
    n_val = int(n * 0.20)

    partes = {
        'train': validos[:n_train],
        'val':   validos[n_train:n_train + n_val],
        'test':  validos[n_train + n_val:],
    }

    for split, archivos_split in partes.items():
        destino = os.path.join(DATASET, split, clase)
        os.makedirs(destino, exist_ok=True)
        for f in archivos_split:
            shutil.copy2(f, destino)

print('Imágenes descartadas por archivo corrupto:', descartadas)
print('DATASET:', DATASET)

In [ ]:
# Control de la estructura final

total = 0
for split in ['train', 'val', 'test']:
    ruta = os.path.join(DATASET, split)
    print(f'[{split}]')
    for clase in sorted(os.listdir(ruta)):
        n = len([f for f in glob.glob(os.path.join(ruta, clase, '*'))
                 if f.lower().endswith(EXTENSIONES)])
        total += n
        print(f'   {clase:20s} {n:5d} imágenes')

print('TOTAL:', total, 'imágenes')

In [ ]:
# Muestra visual: tres imágenes de cada clase

import matplotlib.pyplot as plt

clases_carpeta = sorted(os.listdir(f'{DATASET}/train'))
fig, axes = plt.subplots(len(clases_carpeta), 3, figsize=(11, 4 * len(clases_carpeta)))

for fila, clase in enumerate(clases_carpeta):
    archivos = sorted(glob.glob(f'{DATASET}/train/{clase}/*'))[:3]
    for col, f in enumerate(archivos):
        ax = axes[fila][col]
        ax.imshow(PILImage.open(f).convert('RGB'))
        ax.set_title(clase if col == 1 else '', fontsize=11)
        ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Entrenamiento

Transfer learning desde `yolov8n-cls.pt` (preentrenado en ImageNet).

Diferencias respecto del notebook de la clase:

| Parámetro | Clase | Acá | Motivo |
|---|---|---|---|
| `imgsz` | 128 | 224 | 128 px pierde la textura superficial que distingue óxido de suciedad o sombra |
| `epochs` | 25 | 50 | dataset chico, conviene más pasadas |
| `patience` | — | 10 | corte temprano si la validación deja de mejorar |
| `project` / `name` | — | fijos | evita que las corridas se acumulen en `train2`, `train3`, … |

Con ~525 imágenes de entrenamiento y GPU T4, lleva unos 5-10 minutos.

In [ ]:
%cd {HOME}

RUN = f'{HOME}/runs/corrosion'

!yolo task=classify mode=train model=yolov8n-cls.pt data={DATASET} epochs=50 imgsz=224 batch=16 patience=10 project={HOME}/runs name=corrosion exist_ok=True

In [ ]:
!ls -la {RUN}
!ls -la {RUN}/weights

In [ ]:
# Curvas de entrenamiento a partir de results.csv

import pandas as pd

df = pd.read_csv(f'{RUN}/results.csv')
df.columns = [c.strip() for c in df.columns]
display(df.tail())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df['epoch'], df['train/loss'], label='train loss')
if 'val/loss' in df.columns:
    ax1.plot(df['epoch'], df['val/loss'], label='val loss')
ax1.set_xlabel('época'); ax1.set_ylabel('loss'); ax1.set_title('Pérdida'); ax1.legend()

ax2.plot(df['epoch'], df['metrics/accuracy_top1'], label='top-1')
ax2.set_xlabel('época'); ax2.set_ylabel('accuracy'); ax2.set_title('Exactitud en validación'); ax2.legend()

plt.tight_layout()
plt.show()

print('Mejor top-1 en validación:', round(df['metrics/accuracy_top1'].max(), 4))

## 5. Validación del modelo entrenado

In [ ]:
%cd {HOME}

!yolo task=classify mode=val model={RUN}/weights/best.pt data={DATASET} imgsz=224

## 6. Evaluación sobre el split de test

El descargo de responsabilidad del notebook original señala que la tarea `classify` devuelve una matriz difícil de interpretar y no genera matriz de confusión. Se resuelve acá con un posprocesamiento propio: se recorre `test/`, se toma `probs.top1` de cada predicción y se arman la matriz de confusión y el reporte por clase.

En un caso de inspección, el error caro es el **falso negativo** (una superficie con corrosión clasificada como sana), así que interesa especialmente el *recall* de `con_corrosion`.

In [ ]:
from ultralytics import YOLO
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

model = YOLO(f'{RUN}/weights/best.pt')
print('Clases del modelo:', model.names)

clases = [model.names[i] for i in sorted(model.names)]

y_true, y_pred, rutas, confianzas = [], [], [], []

for idx, clase in enumerate(clases):
    archivos = sorted([f for f in glob.glob(f'{DATASET}/test/{clase}/*')
                       if f.lower().endswith(EXTENSIONES)])
    if not archivos:
        print('Sin imágenes de test para la clase', clase)
        continue
    for res in model.predict(archivos, imgsz=224, verbose=False):
        y_true.append(idx)
        y_pred.append(int(res.probs.top1))
        rutas.append(res.path)
        confianzas.append(float(res.probs.top1conf))

print()
print('Imágenes evaluadas:', len(y_true))
print('Accuracy en test:', round(accuracy_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, target_names=clases, digits=3))

In [ ]:
# Matriz de confusión

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(1.8 * len(clases) + 2.5, 1.6 * len(clases) + 2))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(range(len(clases))); ax.set_xticklabels(clases, rotation=30, ha='right')
ax.set_yticks(range(len(clases))); ax.set_yticklabels(clases)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión — test')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Errores de clasificación: material directo para el análisis del informe

errores = [(r, clases[t], clases[p], c)
           for r, t, p, c in zip(rutas, y_true, y_pred, confianzas) if t != p]

print('Errores:', len(errores), 'de', len(y_true))

for ruta, real, pred, conf in errores[:10]:
    print(f'  {os.path.basename(ruta)[:38]:40s} real={real:16s} pred={pred:16s} conf={conf:.2f}')

if errores:
    n = min(4, len(errores))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (ruta, real, pred, conf) in zip(axes, errores[:n]):
        ax.imshow(PILImage.open(ruta).convert('RGB'))
        ax.set_title(f'real: {real}\npred: {pred} ({conf:.2f})', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Inferencia sobre una imagen nueva

Acá se puede probar el modelo con una foto propia de inspección, aunque el entrenamiento se haya hecho con el dataset público.

In [ ]:
from google.colab import files

subido = files.upload()

for nombre in subido:
    res = model.predict(nombre, imgsz=224, verbose=False)[0]
    probs = res.probs.data.cpu().numpy()
    print(nombre)
    for i in np.argsort(probs)[::-1]:
        print(f'   {clases[i]:20s} {probs[i]:.3f}')
    plt.figure(figsize=(5, 5))
    plt.imshow(PILImage.open(nombre).convert('RGB'))
    plt.title(f'{clases[int(res.probs.top1)]} ({float(res.probs.top1conf):.2f})')
    plt.axis('off')
    plt.show()

In [ ]:
# Descargar los pesos entrenados para la entrega

files.download(f'{RUN}/weights/best.pt')

## 8. Conclusiones

Completar con los resultados obtenidos al correr el notebook:

- **Dataset:** `khoaliamle/Corrosion_Rust` (Hugging Face Hub, MIT), 752 imágenes, 2 clases (`con_corrosion` / `sin_corrosion`), partición propia 70/20/10 con semilla fija.
- **Modelo:** YOLOv8n-cls, transfer learning desde ImageNet, 224×224, 50 épocas con `patience=10` (corte efectivo en la época …).
- **Métricas:** top-1 en validación = … ; accuracy en test = … ; recall de `con_corrosion` = …
- **Análisis de errores:** los fallos se concentran en … (típicamente superficies con pintura naranja, sombras o suciedad que el modelo confunde con óxido, y corrosión incipiente de bajo contraste).
- **Limitaciones:** dataset de uso general, no específico de ductos; imágenes con encuadre, distancia e iluminación heterogéneos; clasificación binaria sin grados de severidad.
- **Mejoras posibles:** ampliar a tres clases por severidad, probar `yolov8s-cls`, subir `imgsz` a 320, aplicar aumentación de brillo y contraste, y reentrenar con imágenes propias de inspección manteniendo criterios uniformes de captura en campo.

**Fuente del dataset:** https://huggingface.co/datasets/khoaliamle/Corrosion_Rust